# บท 04 · Vectorized backtest ที่เทียบกับบัญชีได้

แบบจำลอง SMA Long/Cash จากข้อมูลสังเคราะห์แปดแท่ง รู้สัญญาณหลัง close ซื้อขายที่ open ถัดไป เงินเริ่มต้น 10,000 USD สมมติ ซื้อเศษหุ้นได้ ไม่กู้เงิน ไม่มีดอกเบี้ย และปิดสถานะที่ open แท่ง 8 ตามกฎล่วงหน้า

ต้นทุน 10 bps ต่อมูลค่าธุรกรรมต่อด้านเป็นสมมติฐานการสอน ไม่ใช่อัตรา Webull ไม่มีคำสั่งเครือข่ายหรือการเชื่อมบัญชี


## 1. สภาพแวดล้อม


In [1]:
import platform
import numpy as np
import pandas as pd
print({"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__})


{'python': '3.12.14', 'numpy': '2.5.3', 'pandas': '2.3.2'}


## 2. ข้อมูลในตัวและเวลาที่ใช้

เลขแท่งเป็นลำดับสังเคราะห์ ไม่มีวันที่หรือความถี่ตลาดจริง จึงไม่ annualize ผลตอบแทนหรือ Sharpe


In [2]:
OPEN = [100.0, 101.0, 103.0, 102.0, 105.0, 102.0, 99.0, 101.0]
CLOSE = [100.0, 102.0, 101.0, 104.0, 103.0, 100.0, 98.0, 101.0]
print(pd.DataFrame({"open": OPEN, "close": CLOSE}, index=pd.RangeIndex(1,9,name="bar")).to_string())


      open  close
bar              
1    100.0  100.0
2    101.0  102.0
3    103.0  101.0
4    102.0  104.0
5    105.0  103.0
6    102.0  100.0
7     99.0   98.0
8    101.0  101.0


## 3. แบบจำลองทั้งตาราง

position_open คือสถานะหลังปรับพอร์ต ณ open ส่วนผลตอบแทนจาก open ก่อนหน้าถึง open ปัจจุบันใช้ position_open ของแถวก่อนหน้า การเข้าเต็มพอร์ตต้องหารเงินด้วย 1+c เพื่อเผื่อต้นทุนซื้อ


In [3]:
def run_backtest(short=2, long=3, cost_bps=10.0, initial=10000.0):
    """Signal at close, fill next open, fractional shares, final open liquidation.

    Entries spend all available equity including a fee on trade notional.
    Exits receive notional less the same fee. No shorting, borrowing or interest.
    """
    if any(isinstance(v, bool) or not isinstance(v, int) for v in [short, long]):
        raise ValueError("windows must be integers")
    if not 1 <= short < long < len(OPEN) - 1:
        raise ValueError("require 1 <= short < long < 7")
    if not np.isfinite(cost_bps) or not 0 <= cost_bps < 10000:
        raise ValueError("invalid cost_bps")
    if not np.isfinite(initial) or initial <= 0:
        raise ValueError("initial must be finite and positive")
    cost = cost_bps / 10000
    data = pd.DataFrame({"open": OPEN, "close": CLOSE}, index=pd.RangeIndex(1, 9, name="bar"))
    data["sma_short"] = data["close"].rolling(short, min_periods=short).mean()
    data["sma_long"] = data["close"].rolling(long, min_periods=long).mean()
    data["target_close"] = ((data["sma_short"] > data["sma_long"]) & data["sma_long"].notna()).astype(int)
    data["position_open"] = data["target_close"].shift(1, fill_value=0)
    # At the final open we liquidate by a predeclared horizon rule, ignoring that bar's close.
    data.loc[data.index[-1], "position_open"] = 0
    data["change_open"] = data["position_open"].diff().fillna(data["position_open"])
    data["market_return"] = data["open"].pct_change(fill_method=None).fillna(0)
    held_previous = data["position_open"].shift(1, fill_value=0)
    gross_factor = 1 + held_previous * data["market_return"]
    fee_factor = np.where(data["change_open"] > 0, 1 / (1 + cost),
                          np.where(data["change_open"] < 0, 1 - cost, 1.0))
    data["equity_gross"] = initial * gross_factor.cumprod()
    data["equity_net"] = initial * (gross_factor * fee_factor).cumprod()
    # Comparison starts at the first actionable open, with the same liquidation date and fees.
    benchmark_position = pd.Series(0, index=data.index)
    benchmark_position.iloc[long:-1] = 1
    benchmark_change = benchmark_position.diff().fillna(0)
    benchmark_factor = 1 + benchmark_position.shift(1, fill_value=0) * data["market_return"]
    benchmark_fee = np.where(benchmark_change > 0, 1 / (1 + cost), np.where(benchmark_change < 0, 1 - cost, 1.0))
    data["benchmark_net"] = initial * (benchmark_factor * benchmark_fee).cumprod()
    return data
print("Vectorized model ready.")


Vectorized model ready.


## 4. รันและตรวจเวลา

LONG แรกเกิดหลัง close 3 ซื้อที่ open 4 ราคา 102 กลับ CASH หลัง close 6 แล้วขายที่ open 7 ราคา 99


In [4]:
data = run_backtest()
assert data["position_open"].tolist() == [0, 0, 0, 1, 1, 1, 0, 0]
assert data.index[data["change_open"] > 0].tolist() == [4]
assert data.index[data["change_open"] < 0].tolist() == [7]
print(data[["open", "close", "target_close", "position_open", "equity_gross", "equity_net"]].round(6).to_string())


      open  close  target_close  position_open  equity_gross    equity_net
bar                                                                       
1    100.0  100.0             0              0  10000.000000  10000.000000
2    101.0  102.0             0              0  10000.000000  10000.000000
3    103.0  101.0             1              0  10000.000000  10000.000000
4    102.0  104.0             1              1  10000.000000   9990.009990
5    105.0  103.0             1              1  10294.117647  10283.833813
6    102.0  100.0             0              1  10000.000000   9990.009990
7     99.0   98.0             0              0   9705.882353   9686.489981
8    101.0  101.0             0              0   9705.882353   9686.489981


## 5. ตรวจด้วยบัญชีเงินสดและหุ้นอีกวิธี

ไล่ทีละแท่ง ซื้อ q=E/[P(1+c)] และขายได้เงิน qP(1-c) แล้วเทียบ equity ทุกแถวกับ vectorized model ไม่ใช่ตรวจเฉพาะจุดสุดท้าย


In [5]:
def ledger_check(data, cost_bps=10.0, initial=10000.0):
    """Independent sequential cash/share accounting against the vector model."""
    cost = cost_bps / 10000
    cash, shares = initial, 0.0
    equity = []
    for _, row in data.iterrows():
        price = row["open"]
        if row["position_open"] == 1 and shares == 0:
            shares = cash / (price * (1 + cost))
            cash = 0.0
        elif row["position_open"] == 0 and shares != 0:
            cash = shares * price * (1 - cost)
            shares = 0.0
        equity.append(cash + shares * price)
    np.testing.assert_allclose(equity, data["equity_net"], rtol=1e-12, atol=1e-8)
    return cash, shares
cash, shares = ledger_check(data)
expected = 10000 * 99 / 102 * .999 / 1.001
assert np.isclose(cash, expected)
assert shares == 0
print(f"Cash: {cash:.6f}; shares: {shares:.6f}; direct formula: {expected:.6f}")
print("All rows match the independent cash/share ledger.")


Cash: 9686.489981; shares: 0.000000; direct formula: 9686.489981
All rows match the independent cash/share ledger.


## 6. แยกก่อนต้นทุน หลังต้นทุน และ benchmark

Benchmark ซื้อที่ open 4 ซึ่งเป็นจุดแรกที่กลยุทธ์เริ่มดำเนินการได้ ขายที่ open 8 ด้วยเงินเริ่มต้นและต้นทุนเดียวกัน ทั้งสามค่ารวมช่วง warm-up ที่ยังเป็นเงินสดไว้เหมือนกัน


In [6]:
summary = pd.DataFrame({
    "final_equity": data.iloc[-1][["equity_gross", "equity_net", "benchmark_net"]],
})
summary["return_pct"] = (summary["final_equity"] / 10000 - 1) * 100
assert np.isclose(data["benchmark_net"].iloc[-1], 10000 * 101 / 102 * .999 / 1.001)
print(summary.round(6).to_string())


               final_equity  return_pct
equity_gross    9705.882353   -2.941176
equity_net      9686.489981   -3.135100
benchmark_net   9882.176647   -1.178234


## 7. แบบฝึกหัดต้นทุนและการตรวจหลายพารามิเตอร์

เพิ่มต้นทุนแล้วสัญญาณไม่ควรเปลี่ยน เงินสุทธิของธุรกรรมเดิมต้องไม่สูงขึ้น ตรวจคู่ SMA ที่อนุญาตทุกคู่กับบัญชีอิสระด้วย


In [7]:
sensitivity = []
for bps in [0, 10, 25, 100]:
    result = run_backtest(cost_bps=bps)
    ledger_check(result, cost_bps=bps)
    assert result["position_open"].equals(data["position_open"])
    sensitivity.append({"cost_bps": bps, "final_equity": result["equity_net"].iloc[-1]})
assert all(sensitivity[i]["final_equity"] >= sensitivity[i+1]["final_equity"] for i in range(3))
assert np.isclose(sensitivity[0]["final_equity"], data["equity_gross"].iloc[-1])
checks = 0
for long_window in range(2, 7):
    for short_window in range(1, long_window):
        for bps in [0, 10, 25, 100]:
            ledger_check(run_backtest(short_window, long_window, bps), bps)
            checks += 1
print(pd.DataFrame(sensitivity).round(6).to_string(index=False))
print(f"Independent ledger scenarios checked: {checks}")


 cost_bps  final_equity
        0   9705.882353
       10   9686.489981
       25   9657.473962
      100   9513.686663
Independent ledger scenarios checked: 60


## 8. การแก้ราคาอนาคตต้องไม่เปลี่ยนอดีต

เปลี่ยน close แท่ง 8 เป็น 900 แต่ผลที่ open แท่ง 8 ต้องเท่าเดิม เพราะเป็นจุดที่เราสิ้นสุดก่อนราคาปิดนั้นเกิดขึ้น


In [8]:
original_last_close = CLOSE[-1]
try:
    CLOSE[-1] = 900.0
    future_changed = run_backtest()
finally:
    CLOSE[-1] = original_last_close
pd.testing.assert_series_equal(data["equity_net"], future_changed["equity_net"])
pd.testing.assert_series_equal(data["position_open"], future_changed["position_open"])
print("All open-time positions and equity unchanged after altering close of bar 8.")
print("Last close target before / after:", data["target_close"].iloc[-1], future_changed["target_close"].iloc[-1])


All open-time positions and equity unchanged after altering close of bar 8.
Last close target before / after: 0 1


## 9. เส้นสั้นลงไม่ได้เข้าก่อนทุกครั้ง

เปลี่ยน 2/3 เป็น 1/3 ตอน close 3 ราคา 101 เท่ากับ SMA 3 ทำให้กติกาใหม่ยัง CASH และซื้อครั้งแรกที่ open 5


In [9]:
faster = run_backtest(1, 3)
ledger_check(faster)
assert faster.index[faster["change_open"] > 0].tolist() == [5]
assert faster.index[faster["change_open"] < 0].tolist() == [7]
print(faster[["close", "sma_short", "sma_long", "target_close", "position_open", "equity_net"]].round(6).to_string())


     close  sma_short    sma_long  target_close  position_open    equity_net
bar                                                                         
1    100.0      100.0         NaN             0              0  10000.000000
2    102.0      102.0         NaN             0              0  10000.000000
3    101.0      101.0  101.000000             0              0  10000.000000
4    104.0      104.0  102.333333             1              0  10000.000000
5    103.0      103.0  102.666667             1              1   9990.009990
6    100.0      100.0  102.333333             0              1   9704.581133
7     98.0       98.0  100.333333             0              0   9409.733124
8    101.0      101.0   99.666667             1              0   9409.733124


## ข้อสรุปของการทดลองและแหล่งอ้างอิง

ผลสุทธิของ SMA 2/3 คือ 9,686.489981 หรือ −3.135100% ของเงินต้นสมมติ และต่ำกว่า benchmark ในช่วงนี้ โค้ดที่ตรวจบัญชีผ่านไม่ได้พิสูจน์ความสามารถทำกำไร ข้อมูลแปดแท่งไม่ใช้เลือกกลยุทธ์สำหรับเงินจริง

ยังไม่จำลอง partial fills, order rejection, spread ตามเวลา, market impact, ข้อจำกัดเศษหุ้น, ภาษี หรือ corporate actions

- [pandas 2.3 shift](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.DataFrame.shift.html)
- [pandas 2.3 pct_change](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.Series.pct_change.html)
- Hilpisch บท 4 หน้าเล่ม 90–92 (PDF 110–112): แนวคิด SMA และการวางสถานะตามเวลา โค้ด Long/Cash next-open และบัญชีต้นทุนใน Notebook นี้เขียนใหม่

ตรวจเอกสารวันที่ 11 กันยายน 2026
